# F3 · Risk Score — XGBoost + SHAP

Identifying at-risk students (likely to disengage or underperform) using gradient-boosted trees with SHAP explainability.

In [ ]:
import django, os, sys
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'lumen_project.settings')
django.setup()

## 1. Dataset Description

The risk feature dataset aggregates per-student behavioural signals over a rolling 14-day window.
Each row is one student snapshot; the binary target `at_risk` is 1 if the student failed ≥50 % of flashcards or had zero activity in the last 7 days.

| Column | Description |
|---|---|
| `user_id` | Student identifier |
| `days_active_last_14` | Days with at least one study event |
| `avg_session_duration_min` | Mean session length (minutes) |
| `avg_flashcard_grade` | Mean flashcard score (0–5) |
| `num_chat_messages` | Chat messages sent in window |
| `quiz_pass_rate` | Fraction of quiz attempts passed |
| `streak_days` | Current consecutive-day streak |
| `topics_attempted` | Distinct topics engaged |
| `at_risk` | Binary target (1 = at risk) |

In [ ]:
from analytics.ml.feature_engineering import build_user_risk_features
df = build_user_risk_features()
print('Shape:', df.shape)

# Class balance
balance = df['at_risk'].value_counts(normalize=True).rename({0: 'safe', 1: 'at_risk'})
print('\nClass balance:')
print(balance.to_string())
print(f'\nAt-risk prevalence: {balance.get("at_risk", balance.get(1, 0)):.1%}')

## 2. EDA — Feature Distributions by Risk Label

In [ ]:
import matplotlib.pyplot as plt

feature_cols = [
    'days_active_last_14', 'avg_session_duration_min',
    'avg_flashcard_grade', 'num_chat_messages',
    'quiz_pass_rate', 'streak_days'
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, col in zip(axes, feature_cols):
    groups = [df.loc[df['at_risk'] == v, col].dropna() for v in [0, 1]]
    ax.boxplot(groups, labels=['safe', 'at_risk'], patch_artist=True,
               boxprops=dict(facecolor='lightblue'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(col)
    ax.set_ylabel('Value')

fig.suptitle('Feature Distributions by Risk Label', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../models/03_eda.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Baseline — Logistic Regression

Stratified 5-fold cross-validation with `LogisticRegression(max_iter=1000)` on standardised features provides the baseline ROC-AUC.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

X = df[feature_cols].fillna(df[feature_cols].median())
y = df['at_risk'].astype(int)

lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, class_weight='balanced'))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_auc_scores = cross_val_score(lr_pipe, X, y, cv=cv, scoring='roc_auc')

lr_auc = lr_auc_scores.mean()
print(f'Logistic Regression  ROC-AUC = {lr_auc:.4f}  (±{lr_auc_scores.std():.4f})')

## 4. XGBoost

Gradient-boosted trees with early stopping potential. Same 5-fold CV evaluates ROC-AUC, precision, recall, and F1 at the 0.5 decision threshold.

In [ ]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix

scale_pos = (y == 0).sum() / max((y == 1).sum(), 1)
xgb = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    scale_pos_weight=scale_pos, eval_metric='logloss',
    random_state=42, verbosity=0
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_list, prec_list, rec_list, f1_list = [], [], [], []

for train_idx, val_idx in cv.split(X, y):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    xgb.fit(X_tr, y_tr)
    proba = xgb.predict_proba(X_val)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    auc_list.append(roc_auc_score(y_val, proba))
    prec_list.append(precision_score(y_val, pred, zero_division=0))
    rec_list.append(recall_score(y_val, pred, zero_division=0))
    f1_list.append(f1_score(y_val, pred, zero_division=0))

xgb_auc  = np.mean(auc_list)
xgb_prec = np.mean(prec_list)
xgb_rec  = np.mean(rec_list)
xgb_f1   = np.mean(f1_list)

print(f'XGBoost  ROC-AUC  = {xgb_auc:.4f}')
print(f'XGBoost  Precision= {xgb_prec:.4f}')
print(f'XGBoost  Recall   = {xgb_rec:.4f}')
print(f'XGBoost  F1       = {xgb_f1:.4f}')

# Final confusion matrix on last fold for illustration
print('\nConfusion matrix (last fold):')
print(confusion_matrix(y_val, pred))

## 5. SHAP Summary

SHAP (SHapley Additive exPlanations) decomposes each prediction into per-feature contributions, enabling both global feature importance and local (per-student) explanations.

In [ ]:
import shap
import matplotlib.pyplot as plt

# Fit final model on full dataset
xgb_final = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    scale_pos_weight=scale_pos, eval_metric='logloss',
    random_state=42, verbosity=0
)
xgb_final.fit(X, y)

explainer   = shap.TreeExplainer(xgb_final)
shap_values = explainer.shap_values(X)

# Global feature importance bar plot
shap.summary_plot(shap_values, X, plot_type='bar',
                  feature_names=feature_cols, show=False)
plt.title('SHAP Feature Importance')
plt.tight_layout()
plt.savefig('../models/03_shap.png', dpi=100, bbox_inches='tight')
plt.show()

# Persist model artefact
import joblib
joblib.dump(xgb_final, '../models/risk_score.pkl')
print('Model saved to models/risk_score.pkl')

## 6. Comparison Table

In [ ]:
import pandas as pd

results = pd.DataFrame([
    {
        'Model':     'Logistic Regression',
        'AUC':       round(lr_auc, 4),
        'Precision': 'N/A (CV only)',
        'Recall':    'N/A (CV only)',
        'F1':        'N/A (CV only)',
        'Notes':     'StandardScaler + class_weight=balanced'
    },
    {
        'Model':     'XGBoost',
        'AUC':       round(xgb_auc, 4),
        'Precision': round(xgb_prec, 4),
        'Recall':    round(xgb_rec, 4),
        'F1':        round(xgb_f1, 4),
        'Notes':     'scale_pos_weight, 5-fold StratifiedKFold'
    },
]).set_index('Model')

print(results.to_string())
print(f'\nAUC improvement: +{(xgb_auc - lr_auc):.4f}')

## 7. Limitations

- Synthetic dataset: class separation may be unrealistically clean; expect lower AUC on real student data.
- 14-day rolling window is a design choice; longer windows capture chronic disengagement, shorter windows capture acute drops — the optimal horizon requires A/B testing.
- `at_risk` label constructed heuristically (grade + activity thresholds); ground-truth labels (teacher assessments, final exam outcomes) should replace it.
- No temporal train/test split: test data from the same period as training inflates AUC if students share behaviour patterns across folds.
- SHAP interaction effects are not visualised here; `shap.dependence_plot` should be added for top features.
- Model fairness (across demographic groups) is not yet evaluated; disparate impact analysis is required before production deployment.

## 8. Production Usage

Use `predict_risk(user_id)` from `analytics.ml.risk_score`.

The function:
1. Computes the 14-day feature vector for the given student from live database aggregates.
2. Loads the serialised XGBoost model from `models/risk_score.pkl`.
3. Returns a dict `{risk_score: float, at_risk: bool, top_factors: list[str]}`.

Retrain the model weekly via:

```
python manage.py train_risk_model
```

The management command also writes SHAP values to the database for per-student explainability in the dashboard.

In [ ]:
from analytics.ml.risk_score import predict_risk

# result = predict_risk(user_id=1)
# print(result)
# # {
# #   'risk_score': 0.78,
# #   'at_risk': True,
# #   'top_factors': ['low quiz_pass_rate', 'streak_days=0', 'avg_flashcard_grade=1.2']
# # }

import inspect
print(inspect.signature(predict_risk))